# YOLO CIFAR10: Fine-Tuned Teacher → Pruned Student → Logits Distillation (Classification)

This notebook runs a YOLOv8n-cls **classification** pruning + logits distillation flow on CIFAR10:
1. Load a **CIFAR10-fine-tuned** YOLOv8n-cls teacher (10-class head).
2. Build a `MaseGraph` from the teacher weights and run unstructured pruning to create the student.
3. Distill teacher logits into the pruned student using CIFAR10 mini-batches (images + labels).
4. Evaluate teacher, pruned-only (no KD) student, and distilled student on CIFAR10 validation set.

In [1]:
cls_model_checkpoint = "data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt"
cls_device = "cuda"
cifar_root = "./data"

image_size = 32
cifar_batch_size = 16

cifar_prune_sparsity = 0.5
cifar_kd_alpha = 0.5
cifar_kd_temperature = 3.0
cifar_kd_epochs = 1
lr = 1e-3
seed = 42


## Imports and setup

In [2]:
import copy
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo.yolov8 import MaseYoloClassificationModel, patch_yolo

from mase_kd.core.losses import DistillationLossConfig, hard_label_ce_loss, soft_logit_kl_loss
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")


/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


In [3]:
if cls_device == "cuda" and not torch.cuda.is_available():
    cls_device = "cpu"

torch.manual_seed(seed)
random.seed(seed)

print(f"Using device: {cls_device}")

Using device: cuda


## CIFAR10 image dataset and dataloaders

In [4]:
# Ultralytics trains with mean=(0,0,0), std=(1,1,1) — i.e. bare ToTensor() [0,1].
# Using the standard CIFAR10 mean/std here would shift inputs into ~[-2, +2],
# breaking the teacher's BatchNorm running stats and causing ~15% accuracy.
cifar_transform_train = transforms.Compose([
    transforms.ToTensor(),
])
cifar_transform_eval = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root=cifar_root, train=True, transform=cifar_transform_train, download=True)
val_dataset = datasets.CIFAR10(root=cifar_root, train=False, transform=cifar_transform_eval, download=True)

# drop_last=True keeps batch size fixed for the FX-specialized pruned student graph.

train_loader = DataLoader(
    train_dataset,
    batch_size=cifar_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cifar_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

#print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


## Load CIFAR10-fine-tuned teacher (YOLOv8n-cls, 10 classes)

In [5]:
ultra_teacher = YOLO(cls_model_checkpoint)
nc = ultra_teacher.model.yaml.get("nc", 10)

# Build the teacher using MaseYoloClassificationModel (same as the student path) so
# that its forward returns a flat [batch, nc] tensor.  Using ultra_teacher.model
# directly gives the raw ultralytics ClassificationModel whose train-mode forward
# returns a structured tuple, which _flatten_logits would concatenate into [batch, 2*nc].
teacher_cls_model = MaseYoloClassificationModel(cfg="yolov8n-cls.yaml", nc=nc)
teacher_cls_model = patch_yolo(teacher_cls_model)
teacher_cls_model.load_state_dict(ultra_teacher.model.state_dict())
teacher_cls_model = teacher_cls_model.to(cls_device)
teacher_cls_model.eval()

print(f"Loaded CIFAR10-fine-tuned teacher: {cls_model_checkpoint}")
print(f"Teacher num_classes (nc): {nc}")


Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralyt

## Build MASE graph from teacher weights and prune to create student

In [6]:
# Build a MaseYoloClassificationModel with the CIFAR10 class count, then load
# the fine-tuned teacher weights. We bypass get_yolo_classification_model()
# because it expects the original "yolov8n-cls.pt" name and 1000-class arch.
student_seed_cls_model = MaseYoloClassificationModel(cfg="yolov8n-cls.yaml", nc=nc)
student_seed_cls_model = patch_yolo(student_seed_cls_model)
student_seed_cls_model.load_state_dict(ultra_teacher.model.state_dict())

mg_cls = MaseGraph(student_seed_cls_model)
mg_cls, _ = passes.init_metadata_analysis_pass(mg_cls)

trace_input_cls = torch.randn(cifar_batch_size, 3, image_size, image_size)
placeholder_names_cls = [
    node.name for node in mg_cls.fx_graph.nodes if node.op == "placeholder"
]
dummy_in_cls = {name: trace_input_cls for name in placeholder_names_cls}

mg_cls, _ = passes.add_common_metadata_analysis_pass(
    mg_cls,
    pass_args={
        "dummy_in": dummy_in_cls,
    },
)

pruning_config_cls = {
    "weight": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
    },
}

mg_cls, _ = passes.prune_transform_pass(mg_cls, pass_args=pruning_config_cls)

student_cls_model = mg_cls.model.to(cls_device)

# Snapshot the pruned student before KD training for baseline comparison.
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(cls_device)
pruned_no_kd_model.eval()

print(f"Pruning complete ({cifar_prune_sparsity*100:.0f}% sparsity); using pruned teacher as student.")

Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralyt

INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: model_6_m_0_cv2_conv
INFO     Pruning module: model_6_m_1_cv1_conv
INFO     Pruning module: model_6_m_1_cv2_conv
INFO     Pruning module: model_6_cv2_conv
INFO     Pruning module: model_7_conv
INFO     Pruning module: model_8_cv1_conv
INFO     Pruning module: model_8_m_0_cv1_conv
INFO     P

Pruning complete (50% sparsity); using pruned teacher as student.


## Distill teacher into pruned student on CIFAR10 images + labels

In [7]:
kd_config_cls = DistillationLossConfig(
    alpha=cifar_kd_alpha,
    temperature=cifar_kd_temperature,
)

optimizer = torch.optim.Adam(student_cls_model.parameters(), lr=lr)

distiller_cls = YOLOLogitsDistiller(
    teacher=teacher_cls_model,
    student=student_cls_model,
    kd_config=kd_config_cls,
    device=cls_device,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_train_epochs=cifar_kd_epochs,
    eval_teacher=True,
)

loss_history = distiller_cls.train()


Epoch 1/1
  Batch 0001/3125 | total=1.407999 | hard=1.022820 | soft=1.793178
  Batch 0010/3125 | total=2.087477 | hard=1.932620 | soft=2.242334
  Batch 0020/3125 | total=1.460036 | hard=1.125291 | soft=1.794780
  Batch 0030/3125 | total=1.749813 | hard=1.507934 | soft=1.991691
  Batch 0040/3125 | total=1.904917 | hard=1.639664 | soft=2.170171
  Batch 0050/3125 | total=1.450739 | hard=1.526674 | soft=1.374804
  Batch 0060/3125 | total=2.322367 | hard=2.405303 | soft=2.239431
  Batch 0070/3125 | total=1.103956 | hard=0.990730 | soft=1.217182
  Batch 0080/3125 | total=1.297202 | hard=0.936511 | soft=1.657893
  Batch 0090/3125 | total=1.256505 | hard=1.196971 | soft=1.316040
  Batch 0100/3125 | total=1.033758 | hard=1.181093 | soft=0.886424
  Batch 0110/3125 | total=1.607742 | hard=1.534573 | soft=1.680910
  Batch 0120/3125 | total=2.046727 | hard=1.916972 | soft=2.176481
  Batch 0130/3125 | total=1.329668 | hard=1.478282 | soft=1.181053
  Batch 0140/3125 | total=2.374025 | hard=2.144163 |

## Evaluation: teacher vs pruned-only vs distilled student (CIFAR10 validation)

In [8]:
import time

# evaluate_model_on_cifar10_val is kept as a standalone helper for the
# pruned-no-KD baseline, which is not managed by the distiller.
@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device):
    """Evaluate a model on the full validation loader. Returns top-1 accuracy, CE loss, and timing."""
    model.eval()
    batches = 0
    samples = 0
    total_forward_ms = 0.0
    correct_top1 = 0
    total_ce_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        batches += 1
        samples += images.shape[0]

        max_label = int(labels.max().item())
        if logits.shape[1] > max_label:
            preds = logits.argmax(dim=1)
            correct_top1 += int((preds == labels).sum().item())
            total_ce_loss += hard_label_ce_loss(logits, labels).item()

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "top1_acc": correct_top1 / max(samples, 1),
        "avg_ce_loss": total_ce_loss / max(batches, 1),
    }

# --- Training loss summary ---
if "loss_history" in globals() and len(loss_history) > 0:
    print(f"Recorded {len(loss_history)} KD training batches")
    print(f"  First loss: {loss_history[0]:.6f}")
    print(f"  Last loss:  {loss_history[-1]:.6f}")
else:
    print("No training loss history found in current kernel session.")

# --- Evaluate teacher + distilled student via the distiller ---
print("\n--- CIFAR10 Validation Results ---\n")

eval_results = distiller_cls.evaluate()
teacher_metrics = eval_results.get("teacher")
student_metrics = eval_results["student"]
val_kd_loss = eval_results["val_kd_loss"]
kd_batches = eval_results["kd_batches"]

# --- Pruned-no-KD baseline (evaluated standalone, not via the distiller) ---
pruned_no_kd_metrics = None
if "pruned_no_kd_model" in globals():
    pruned_no_kd_metrics = evaluate_model_on_cifar10_val(pruned_no_kd_model, val_loader, cls_device)

# --- Print results ---
if teacher_metrics is not None:
    print(f"Teacher (fine-tuned on CIFAR10):")
    print(
        f"  top1_acc={teacher_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={teacher_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={teacher_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={teacher_metrics['samples']}"
    )

if pruned_no_kd_metrics is not None:
    print(f"\nPruned student (NO distillation, {cifar_prune_sparsity*100:.0f}% sparsity):")
    print(
        f"  top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={pruned_no_kd_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={pruned_no_kd_metrics['samples']}"
    )

print(f"\nDistilled student (pruned + {cifar_kd_epochs} epochs, alpha={cifar_kd_alpha}, T={cifar_kd_temperature}):")
print(
    f"  top1_acc={student_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={student_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={student_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={student_metrics['samples']}"
)

print(f"\nValidation KD loss (teacher vs distilled student, T={cifar_kd_temperature}): "
      f"{val_kd_loss:.6f} over {kd_batches} batches")

# --- Summary table ---
print("\n--- Summary ---")
print(f"{'Model':<40} {'Top-1 Acc':>10} {'CE Loss':>10}")
print(f"{'─'*40} {'─'*10} {'─'*10}")
if teacher_metrics is not None:
    print(f"{'Teacher (fine-tuned)':<40} {teacher_metrics['top1_acc']*100:>9.2f}% {teacher_metrics['avg_ce_loss']:>10.4f}")
if pruned_no_kd_metrics is not None:
    print(f"{'Pruned student (no KD)':<40} {pruned_no_kd_metrics['top1_acc']*100:>9.2f}% {pruned_no_kd_metrics['avg_ce_loss']:>10.4f}")
print(f"{'Distilled student (pruned + KD)':<40} {student_metrics['top1_acc']*100:>9.2f}% {student_metrics['avg_ce_loss']:>10.4f}")


Recorded 3125 KD training batches
  First loss: 1.407999
  Last loss:  1.129928

--- CIFAR10 Validation Results ---

Teacher (fine-tuned on CIFAR10):
  top1_acc=77.58% | CE_loss=1.7555 | fwd_ms/batch=6.26 | samples=10000

Pruned student (NO distillation, 50% sparsity):
  top1_acc=36.29% | CE_loss=1.8760 | fwd_ms/batch=15.95 | samples=10000

Distilled student (pruned + 1 epochs, alpha=0.5, T=3.0):
  top1_acc=65.45% | CE_loss=1.0021 | fwd_ms/batch=21.63 | samples=10000

Validation KD loss (teacher vs distilled student, T=3.0): 1.028319 over 625 batches

--- Summary ---
Model                                     Top-1 Acc    CE Loss
──────────────────────────────────────── ────────── ──────────
Teacher (fine-tuned)                         77.58%     1.7555
Pruned student (no KD)                       36.29%     1.8760
Distilled student (pruned + KD)              65.45%     1.0021


# Test

In [7]:
# Build a MaseYoloClassificationModel with the CIFAR10 class count, then load
# the fine-tuned teacher weights. We bypass get_yolo_classification_model()
# because it expects the original "yolov8n-cls.pt" name and 1000-class arch.
student_seed_cls_model = MaseYoloClassificationModel(cfg="yolov8n-cls.yaml", nc=nc)
student_seed_cls_model = patch_yolo(student_seed_cls_model)
student_seed_cls_model.load_state_dict(ultra_teacher.model.state_dict())

mg_cls = MaseGraph(student_seed_cls_model)
mg_cls, _ = passes.init_metadata_analysis_pass(mg_cls)

trace_input_cls = torch.randn(cifar_batch_size, 3, image_size, image_size)
placeholder_names_cls = [
    node.name for node in mg_cls.fx_graph.nodes if node.op == "placeholder"
]
dummy_in_cls = {name: trace_input_cls for name in placeholder_names_cls}

mg_cls, _ = passes.add_common_metadata_analysis_pass(
    mg_cls,
    pass_args={
        "dummy_in": dummy_in_cls,
    },
)

pruning_config_cls = {
    "weight": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
    },
}

mg_cls, _ = passes.prune_transform_pass(mg_cls, pass_args=pruning_config_cls)

student_cls_model = mg_cls.model.to(cls_device)

# Snapshot the pruned student before KD training for baseline comparison.
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(cls_device)
pruned_no_kd_model.eval()

print(f"Pruning complete ({cifar_prune_sparsity*100:.0f}% sparsity); using pruned teacher as student.")

Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralyt

INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: model_6_m_0_cv2_conv
INFO     Pruning module: model_6_m_1_cv1_conv
INFO     Pruning module: model_6_m_1_cv2_conv
INFO     Pruning module: model_6_cv2_conv
INFO     Pruning module: model_7_conv
INFO     Pruning module: model_8_cv1_conv
INFO     Pruning module: model_8_m_0_cv1_conv
INFO     P

Pruning complete (5% sparsity); using pruned teacher as student.


In [10]:
# Only check Pruned student (no KD) model

import time

# evaluate_model_on_cifar10_val is kept as a standalone helper for the
# pruned-no-KD baseline, which is not managed by the distiller.
@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device):
    """Evaluate a model on the full validation loader. Returns top-1 accuracy, CE loss, and timing."""
    model.eval()
    batches = 0
    samples = 0
    total_forward_ms = 0.0
    correct_top1 = 0
    total_ce_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        batches += 1
        samples += images.shape[0]

        max_label = int(labels.max().item())
        if logits.shape[1] > max_label:
            preds = logits.argmax(dim=1)
            correct_top1 += int((preds == labels).sum().item())
            total_ce_loss += hard_label_ce_loss(logits, labels).item()

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "top1_acc": correct_top1 / max(samples, 1),
        "avg_ce_loss": total_ce_loss / max(batches, 1),
    }

# --- Evaluate teacher + distilled student via the distiller ---
print("\n--- CIFAR10 Validation Results ---\n")

# --- Pruned-no-KD baseline (evaluated standalone, not via the distiller) ---
pruned_no_kd_metrics = None

pruned_no_kd_metrics = evaluate_model_on_cifar10_val(pruned_no_kd_model, val_loader, cls_device)

# --- Print results ---

if pruned_no_kd_metrics is not None:
    print(f"\nPruned student (NO distillation, {cifar_prune_sparsity*100:.0f}% sparsity):")
    print(
        f"  top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={pruned_no_kd_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={pruned_no_kd_metrics['samples']}"
    )

# --- Summary table ---
print("\n--- Summary ---")
print(f"{'Model':<40} {'Top-1 Acc':>10} {'CE Loss':>10}")
print(f"{'─'*40} {'─'*10} {'─'*10}")
if pruned_no_kd_metrics is not None:
    print(f"{'Pruned student (no KD)':<40} {pruned_no_kd_metrics['top1_acc']*100:>9.2f}% {pruned_no_kd_metrics['avg_ce_loss']:>10.4f}")



--- CIFAR10 Validation Results ---


Pruned student (NO distillation, 50% sparsity):
  top1_acc=36.26% | CE_loss=1.8836 | fwd_ms/batch=17.82 | samples=10000

--- Summary ---
Model                                     Top-1 Acc    CE Loss
──────────────────────────────────────── ────────── ──────────
Pruned student (no KD)                       36.26%     1.8836
